In [10]:
"""
Install once:  pip install pybaseball pandas numpy matplotlib

Shifts were allowed in 2022 and restricted starting in 2023.
We pull a sample window from each season 
(full seasons are ~700k rows each as each baseball game is hundreds of distinct statstical events and each one is recorded in full by google statcast)
and compare BABIP (batting average on balls in play) before vs. after.
this code was created with the help of ai but i read thru it and described it with my own comments so i have a good idea of how it functions 
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pybaseball import statcast, cache #pybaseball is a python library that makes it extremely easy to pull from MLB's official statcast data.

cache.enable()  # saves downloads locally so re-runs are fast

# if we pull from the same month in each year we can avoid weather being a major factor
# we can start with just a few months from each year to keep it simple and expand if needed
SEASONS = {
    2022: ("2022-04-01", "2022-05-31"),  # shifts allowed
    2023: ("2023-04-01", "2023-05-31"),  # shift restrictions begin
    2024: ("2024-04-01", "2024-05-31"),  # restrictions continue
}
CSV_PATH = "statcast_sample.csv"

KEEP_COLS = [ # these are the variables we will need to see if the shift ban actually affected BABIP
    "game_date", "game_year", "batter", "player_name", "stand", "events",
    "type", "bb_type", "hc_x", "hc_y", "launch_speed", "launch_angle",
] 

try:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df):,} rows from {CSV_PATH}") # try to load cached csv
except FileNotFoundError: # if you cant download data we selected earlier
    frames = []
    for year, (start, end) in SEASONS.items():
        print(f"Downloading {year}: {start} to {end} (this can take a few minutes)...")
        chunk = statcast(start_dt=start, end_dt=end)
        frames.append(chunk[KEEP_COLS])
    df = pd.concat(frames, ignore_index=True)
    df.to_csv(CSV_PATH, index=False)
    print(f"Saved {len(df):,} rows to {CSV_PATH}") # print to file

# BABIP does not include home runs or bunts so we exclude those from the dataset

bip = df[(df["type"] == "X") & df["events"].notna()].copy()
bip = bip[~bip["events"].isin(["home_run", "sac_bunt", "sac_bunt_double_play",
                               "catcher_interf"])]
bip = bip.dropna(subset=["stand", "hc_x", "hc_y"])

HITS = ["single", "double", "triple"]
bip["is_hit"] = bip["events"].isin(HITS).astype(int)

# spray angle is where the ball was hit in a 2d plane from home plate. positive data is to the right of the field
# (Home plate is roughly at hc_x=125.42, hc_y=198.27 in Statcast coordinates.)
bip["spray_angle"] = np.degrees(
    np.arctan((bip["hc_x"] - 125.42) / (198.27 - bip["hc_y"]))
)

# A "pulled" ball goes to right field for lefties, left field for righties.
bip["pulled"] = (
    ((bip["stand"] == "L") & (bip["spray_angle"] > 15))
    | ((bip["stand"] == "R") & (bip["spray_angle"] < -15))
)
# Save the cleaned balls-in-play dataset to a CSV file
bip.to_csv('cleaned_statcast_bip.csv', index=False)
print("Cleaned data successfully saved to 'cleaned_statcast_bip.csv'")

This is a large query, it may take a moment to complete


100%|██████████| 61/61 [00:05<00:00, 10.29it/s]


This is a large query, it may take a moment to complete


100%|██████████| 61/61 [00:06<00:00,  9.90it/s]


This is a large query, it may take a moment to complete


100%|██████████| 61/61 [00:05<00:00, 11.49it/s]


Saved 707,966 rows to statcast_sample.csv
Cleaned data successfully saved to 'cleaned_statcast_bip.csv'


In [12]:
# summary
# the dataset with all statcast data is really big (mine is 700,000+ rows without cleaning) so it's really good that we cleaned it.
# the cleaned dataset is hard to make any specific observations about without further anaylsis. 
# it contains all non-homerun and non-bunt balls hit into play in the months of april and may of 2022, 23, and 24.

# reflection
# i think this is a good dataset choice as the statcast data gives you a really good platform to dynamically generate a dataset based on specific perameters
# i also think baseball is a really good sport for this because other big league sports do not have quite the level of advanced metrics that baseball does
# this is very fun to investigate